# Phase 4: Predictive Analysis

## Hypothesis: 
Future glucose (e.g., 30–60 minutes out) can be predicted more accurately by combining recent glucose trends with recent physical activity and ingested insulin/carbs, rather than looking at glucose trends alone.

## Model Selection: 
While LSTMs are great for time series, XGBoost or LightGBM are vastly superior for hackathons. They train instantly, handle tabular lag features beautifully, and provide feature importance out-of-the-box.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def train_predictive_model(df):
    df['glucose_in_30m'] = df['glucose'].shift(-6)
    df['glucose_in_60m'] = df['glucose'].shift(-12)
    
    # Calculate a rolling sum of active insulin/carbs over the last 2 hours
    df['carbs_last_2h'] = df['carb_input'].rolling(window=24, min_periods=1).sum()
    df['bolus_last_2h'] = df['bolus_volume_delivered'].rolling(window=24, min_periods=1).sum()
    df['steps_last_2h'] = df['steps'].rolling(window=24, min_periods=1).sum()
    
    # Feature Engineering: Lags and Rolling Windows
    df['glucose_lag_1'] = df['glucose'].shift(1) # 5 mins ago
    df['glucose_lag_3'] = df['glucose'].shift(3) # 15 mins ago
    df['glucose_rate_of_change'] = df['glucose'] - df['glucose_lag_3']
    
    # Target: Predict glucose 45 minutes in the future (9 steps * 5 mins)
    df['target_glucose_45m'] = df['glucose'].shift(-9)
    
    # Drop NaNs created by shifting
    model_df = df.dropna()
    
    features = ['glucose', 'glucose_lag_1', 'glucose_lag_3', 'glucose_rate_of_change', 
                'carbs_last_2h', 'bolus_last_2h', 'steps_last_2h', 'heart_rate']
    
    X = model_df[features]
    y = model_df['target_glucose_45m']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False) # MUST be False for time series
    
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    
    print(f"Mean Absolute Error predicting 45 mins ahead: {mae:.2f} mg/dL")
    return model, features

In [9]:
import import_ipynb
from Data_Preprocessing_Cleaning import read_and_preprocess_data

In [10]:
cleaned_df = read_and_preprocess_data('/Users/jputha177@cable.comcast.com/Downloads/HUPA0002P.csv')

In [11]:
train_predictive_model(cleaned_df)

Mean Absolute Error predicting 45 mins ahead: 17.22 mg/dL


(RandomForestRegressor(max_depth=10, random_state=42),
 ['glucose',
  'glucose_lag_1',
  'glucose_lag_3',
  'glucose_rate_of_change',
  'carbs_last_2h',
  'bolus_last_2h',
  'steps_last_2h',
  'heart_rate'])